# Drishti — VizWiz Baseline (stock model, no fine-tuning)

**Goal:** the baseline number. Run the stock VLM on N VizWiz-val questions and score
with the **official VizWiz accuracy metric**: `acc = mean( min(#matching-humans / 3, 1) )`
over the 10 crowd answers per question.

**RESULT (2026-08-02)** — everything later must beat this row:

| Model | N | Overall acc | Answerable acc | Unanswerable acc | s/answer |
|---|---|---|---|---|---|
| stock SmolVLM-Instruct | 500 | **0.308** | 0.310 (n=256) | 0.306 (n=244) | 1.21 |

See the analysis cell at the bottom — the actionable finding is that **49% of the eval set is
unanswerable** and the model guesses instead of declining, which is where nearly all the
available accuracy lives.

**Base model: SmolVLM, not Moondream-2.** The notebook-00 spike measured both on real
VizWiz photos and SmolVLM won on the axes that matter here:

| | Moondream-2 | SmolVLM |
|---|---|---|
| Latency | ~4.4 s/answer | **~1.75 s** (measured 1.21 s over 500) |
| Answer style | verbose paragraphs | **terse** |
| Loading | `trust_remote_code` — breaks on transformers v5 | **native transformers classes** |

Terseness is not cosmetic: VizWiz scores by **exact match** against short crowd answers, so
a verbose-but-correct answer scores ~0. Moondream also fabricated a drug name and ingredient
list when asked about a medicine — the finding that motivates the `app/drug_db.py` guardrail.

*Caveat worth reporting honestly:* Moondream read a book title correctly (`Dog Years: A
Memoir`) where SmolVLM did not (`Twelve years`). SmolVLM was selected on speed, terseness and
robustness — not on being better at every question type.

Moondream remains selectable below for comparison, but it requires `transformers<5`.

Colab: `Runtime → T4 GPU → Run all` (~15 min for N=500 with SmolVLM).

In [ ]:
%pip install -q -U transformers accelerate datasets einops
import torch, time, json, re, string, transformers
from itertools import islice
from datasets import load_dataset

print('transformers:', transformers.__version__)

# --- model selection -----------------------------------------------------------------
# SmolVLM uses native transformers classes, so it is immune to the trust_remote_code
# breakage that killed Moondream on transformers v5. It was also 2.5x faster and terser
# in the notebook-00 spike -- see this notebook's header for the comparison.
MODEL_ID = 'HuggingFaceTB/SmolVLM-Instruct'
# MODEL_ID = 'vikhyatk/moondream2'   # comparison only; REQUIRES pip install "transformers<5"

# Moondream needs transformers 4.x; fail fast with an actionable message instead of an
# obscure AttributeError two cells later.
if 'moondream' in MODEL_ID.lower() and int(transformers.__version__.split('.')[0]) >= 5:
    raise RuntimeError(
        f"Moondream needs transformers<5 but {transformers.__version__} is loaded.\n"
        f"Change the install line above to: %pip install -q 'transformers<5' ...\n"
        f"then Runtime -> Restart session -> Run all (a pip downgrade cannot replace an\n"
        f"already-imported module, which is why restarting is required)."
    )

N_SAMPLES = 500          # increase to full val (4319) for the report if time allows
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '| model:', MODEL_ID)

# VizWiz-specific prompt: the metric rewards saying 'unanswerable' when the photo is unusable
PROMPT_SUFFIX = (" Answer in one to three words. If the question cannot be answered"
                 " from the image, answer exactly: unanswerable")

In [ ]:
stream = load_dataset('lmms-lab/VizWiz-VQA', split='val', streaming=True)
data = list(islice(stream, N_SAMPLES))

def gt_answers(sample):
    """Normalize: answers may be list[str] or list[{'answer': ...}]."""
    ans = sample['answers']
    return [a['answer'] if isinstance(a, dict) else a for a in ans]

print(len(data), 'samples ·', 'fields:', list(data[0].keys()))
print('example answers:', gt_answers(data[0])[:4])

In [ ]:
# Loader dispatches on model family so the rest of the notebook is model-agnostic.
# SmolVLM -> native transformers classes. Moondream -> trust_remote_code (transformers 4.x).

if 'moondream' in MODEL_ID.lower():
    from transformers import AutoModelForCausalLM, AutoTokenizer

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, trust_remote_code=True, torch_dtype=torch.float16, device_map=DEVICE)
    _tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

    def answer(img, question):
        prompt = question + PROMPT_SUFFIX
        try:
            return model.query(img, prompt)['answer']
        except AttributeError:
            return model.answer_question(model.encode_image(img), prompt, _tok)

else:
    from transformers import AutoProcessor
    try:  # AutoModelForVision2Seq is deprecated in favour of this in transformers 5
        from transformers import AutoModelForImageTextToText as _VisionSeq
    except ImportError:
        from transformers import AutoModelForVision2Seq as _VisionSeq

    processor = AutoProcessor.from_pretrained(MODEL_ID)
    model = _VisionSeq.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map=DEVICE)
    model.eval()

    def answer(img, question):
        msgs = [{'role': 'user',
                 'content': [{'type': 'image'},
                             {'type': 'text', 'text': question + PROMPT_SUFFIX}]}]
        prompt = processor.apply_chat_template(msgs, add_generation_prompt=True)
        inputs = processor(text=prompt, images=[img.convert('RGB')], return_tensors='pt').to(DEVICE)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=24, do_sample=False)
        text = processor.batch_decode(out, skip_special_tokens=True)[0]
        return text.split('Assistant:')[-1].strip()

# smoke-test on one sample before committing to the full 500-question run
_s = data[0]
_t0 = time.time()
print('Q :', _s['question'])
print('A :', answer(_s['image'], _s['question']), f'({time.time() - _t0:.1f}s)')
print('GT:', gt_answers(_s)[:4])

In [ ]:
from tqdm.auto import tqdm

results = []
for s in tqdm(data):
    t0 = time.time()
    try:
        pred = answer(s['image'], s['question'])
    except Exception as e:
        pred = f'__error__ {e}'
    results.append({'question': s['question'], 'prediction': pred,
                    'answers': gt_answers(s), 'latency_s': round(time.time() - t0, 2)})
print('done:', len(results))

In [ ]:
# Official-style VQA answer normalization (simplified) + VizWiz accuracy
ARTICLES = {'a', 'an', 'the'}
def norm(t):
    t = t.lower().strip()
    t = t.translate(str.maketrans('', '', string.punctuation))
    return ' '.join(w for w in t.split() if w not in ARTICLES)

def vizwiz_acc(pred, answers):
    p = norm(pred)
    matches = sum(norm(a) == p for a in answers)
    return min(matches / 3.0, 1.0)

for r in results:
    r['acc'] = vizwiz_acc(r['prediction'], r['answers'])
    r['unanswerable_gt'] = sum(norm(a) == 'unanswerable' for a in r['answers']) >= 5

overall = sum(r['acc'] for r in results) / len(results)
ans_set = [r for r in results if not r['unanswerable_gt']]
una_set = [r for r in results if r['unanswerable_gt']]
lat = sum(r['latency_s'] for r in results) / len(results)

print(f'BASELINE — {MODEL_ID} on {len(results)} VizWiz-val samples')
print(f'  overall accuracy      : {overall:.3f}')
print(f'  answerable subset     : {sum(r["acc"] for r in ans_set)/max(len(ans_set),1):.3f}  (n={len(ans_set)})')
print(f'  unanswerable subset   : {sum(r["acc"] for r in una_set)/max(len(una_set),1):.3f}  (n={len(una_set)})')
print(f'  mean latency (T4 GPU) : {lat:.2f}s/answer')

In [ ]:
# Save results + inspect worst failures (these guide fine-tuning and prompt fixes)
import pandas as pd
df = pd.DataFrame([{k: v for k, v in r.items() if k != 'answers'} | 
                   {'gt_sample': '; '.join(r['answers'][:3])} for r in results])
df.to_csv('vizwiz_baseline_results.csv', index=False)
print('saved vizwiz_baseline_results.csv — download and commit to eval/results/')
df[df.acc == 0].head(15)[['question', 'prediction', 'gt_sample']]

## Baseline result — 2026-08-02

| Model | N | Overall | Answerable | Unanswerable | s/answer |
|---|---|---|---|---|---|
| **stock SmolVLM-Instruct** | 500 | **0.308** | 0.310 (n=256) | 0.306 (n=244) | 1.21 (T4) |

This is the number LoRA fine-tuning must beat. Same N, same prompt, same metric when repeated.

---

### The headline finding: unanswerable detection is where the accuracy is

**49% of VizWiz-val is unanswerable** (244/500) — photos too blurry, too dark, or framed
wrong, which is exactly what happens when the photographer cannot see. The model scores only
0.306 there because it *guesses instead of declining*:

| Question | Predicted | Ground truth |
|---|---|---|
| "What does this sign say?" | `Pizza express` | unanswerable |
| "Is this shampoo or conditioner?" | `Shampoo` | unanswerable |
| "What does my eye look like?" | `Blurry` | unanswerable |
| "This piece of mail... where is it from?" | `Malaysia` | unanswerable |
| "What the screen says?" | `Windows` | unanswerable |

Where the headroom actually is, holding the other subset fixed:

| Change | Overall | Delta |
|---|---|---|
| unanswerable 0.31 → 0.60 | 0.452 | **+0.143** |
| unanswerable 0.31 → 0.80 | 0.549 | **+0.241** |
| unanswerable 0.31 → 1.00 | 0.647 | **+0.339** |
| answerable 0.31 → 0.50 | 0.405 | +0.097 |

**Fine-tuning should target abstention first, not general VQA skill.** Doubling the
unanswerable rate is worth more than a 60% relative gain in answering ability.

This is also a *safety* result, not just a metric one: a blind user acting on a confidently
hallucinated answer about a blurry photo is the exact failure `app/drug_db.py` guards
against in medicine mode. The same principle now has quantitative backing for every mode.

### Three failure patterns for the report

1. **Over-answering / no abstention** — dominant, ~49% of the eval set. See above.
2. **Fine-grained OCR misses** — `545` vs `1545` (tag number); `Twelve years` vs
   `dog years` (book title, which Moondream-2 read correctly in the notebook-00 spike).
   Motivates the routing design: send text-reading to PaddleOCR, not the VLM.
3. **Question-form misreads** — "Which one is the blue one?" → `Blue` instead of `right`.
   The model answers the attribute rather than the location being asked for.

### Next

1. Download `vizwiz_baseline_results.csv` → `eval/results/`, run `python eval/analyze_results.py`
2. Update `docs/BUILD_PLAN.md` (done) and re-run this notebook after fine-tuning
3. Fine-tuning must weight abstention examples — see Phase 3 in the build plan